# Bài 12 · Trực quan hoá cơ bản

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Chọn dạng biểu đồ theo **câu hỏi** (line/bar/hist/scatter) và vẽ bằng `fig, ax`.
2. Hoàn thiện một hình "tự đứng được": title-thông-điệp, nhãn + đơn vị, chú nguồn, nhấn đúng chỗ.
3. Xuất hình từ pipeline (`savefig` vào `figures/`) với phong cách đồng bộ (`rcParams`).
4. Nhận diện và tránh các chiêu "nói dối bằng biểu đồ" (trục cắt gốc, khác thang…).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Style đồng bộ cho CẢ notebook — khai một lần
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "font.size": 12,
})
XANH, CAM = "#1E93AB", "#E8890C"
Path("figures").mkdir(exist_ok=True)

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29"
df = pd.read_csv(f"{BASE}/visualisations/listings.csv")
rv = pd.read_csv(f"{BASE}/visualisations/reviews.csv", parse_dates=["date"])
print(df.shape, rv.shape)

## 1. Line — diễn biến theo thời gian

In [ ]:
thang = rv.set_index("date").resample("ME").size().iloc[:-1]   # bỏ kỳ cụt (buổi 8!)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(thang.index, thang.values, color=XANH, lw=2)
ax.set_title("Thị trường Santiago phục hồi và tăng tốc sau COVID", loc="left", fontweight="bold")
ax.set_ylabel("số review / tháng")
ax.text(1, -0.18, "Nguồn: Inside Airbnb, 29/06/2026", transform=ax.transAxes,
        ha="right", fontsize=9, color="#777")
fig.savefig("figures/reviews_theo_thang.png", dpi=150, bbox_inches="tight")
plt.show()

Ba thứ làm hình này "tự đứng được": title nói **thông điệp** (không phải "Biểu đồ số review"),
trục y có **đơn vị**, chân hình có **nguồn + ngày snapshot**.

## 2. Bar ngang — so sánh nhóm

In [ ]:
tk = df.groupby("neighbourhood")["price"].agg(median="median", n="size")
top = tk[tk["n"] >= 500].nlargest(8, "median").sort_values("median")

fig, ax = plt.subplots(figsize=(8.6, 4))
mau = [CAM if q == top["median"].idxmax() else XANH for q in top.index]
bars = ax.barh(top.index, top["median"], color=mau, height=0.6)
ax.bar_label(bars, [f" {v/1000:,.0f}k" for v in top["median"]], fontsize=10)
ax.set_title("Lo Barnechea bỏ xa phần còn lại về giá trung vị", loc="left", fontweight="bold")
ax.set_xlabel("giá trung vị (CLP/đêm), quận ≥ 500 listing")
ax.grid(axis="y", alpha=0)
fig.savefig("figures/gia_theo_quan.png", dpi=150, bbox_inches="tight")
plt.show()

Vì sao **ngang**? Tên quận dài — bar ngang cho chữ nằm thẳng, khỏi xoay đầu.
Vì sao chỉ **một thanh cam**? Nhấn đúng một thứ; tô mỗi thanh một màu là trang trí, không phải thông tin.

## 3. Hist — hình dáng phân phối

In [ ]:
gia = df.loc[df["price"] > 0, "price"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(gia, bins=55, color=XANH)
axes[0].set_title("Thang thường: chỉ thấy một cột", fontsize=11)
axes[1].hist(np.log10(gia), bins=55, color=CAM)
axes[1].set_title("Thang log10: thấy cả phân phối", fontsize=11)
for ax in axes:
    ax.set_ylabel("số listing")
axes[1].set_xlabel("log10(giá)")
plt.tight_layout(); plt.show()

## 4. Scatter — "bản đồ của người nghèo"

In [ ]:
s = df.dropna(subset=["price"]).sample(6000, random_state=1)
mau = np.where(s["price"] > s["price"].median(), CAM, XANH)

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.scatter(s["longitude"], s["latitude"], s=6, c=mau, alpha=0.45, linewidths=0)
ax.set_aspect("equal")
ax.set_title("Nửa đắt của thị trường (cam) dồn về đông bắc", loc="left", fontweight="bold")
ax.set_xlabel("kinh độ"); ax.set_ylabel("vĩ độ")
plt.tight_layout(); plt.show()

Chỉ cần lat/lon + màu nhị phân là thấy cấu trúc không gian. `alpha=0.45` + chấm nhỏ cứu 6.000
điểm khỏi dẫm nhau. (Bản đồ thật với ranh giới quận: buổi 13.)

## 5. Nói dối bằng biểu đồ — tự tay làm để tự phòng

In [ ]:
gia_loai = df.groupby("room_type")["price"].median() / 1000
gia_loai = gia_loai[["Private room", "Entire home/apt", "Hotel room"]]

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.5))
for ax, y0, ten, mau_t in [(axes[0], 30, "Trục cắt từ 30 — chênh lệch bị thổi phồng", "#c0392b"),
                            (axes[1], 0, "Trục từ 0 — tỷ lệ thật", "#2E8B57")]:
    bars = ax.bar(["Phòng riêng", "Nguyên căn", "Khách sạn"], gia_loai.values, color=XANH, width=0.55)
    ax.set_ylim(y0, 130)
    ax.set_title(ten, color=mau_t, fontsize=11, fontweight="bold")
    ax.set_ylabel("giá trung vị (nghìn CLP)")
    ax.grid(axis="x", alpha=0)
plt.tight_layout(); plt.show()

Cùng số liệu, hai ấn tượng khác hẳn. **Bar chart: trục y bắt đầu từ 0** — không thương lượng.
(Line chart được phép zoom trục y, vì line thể hiện *biến thiên*, không phải *độ lớn* bằng chiều cao.)

## 6. Bài tập tại lớp

### Bài 1 — Từ nháp đến xuất bản

Cell dưới là bản nháp một dòng. Nâng cấp nó thành hình xuất bản được: `fig, ax` + title thông điệp
+ nhãn/đơn vị + nguồn + lưu vào `figures/`. Dùng checklist: *che title đi, người lạ nhìn 10 giây
có hiểu đúng không?*

In [ ]:
# Bản nháp:
df["room_type"].value_counts().plot.barh()
plt.show()

# TODO: bản xuất bản của bạn
ty_le = df["room_type"].value_counts(normalize=True).sort_values() * 100
fig, ax = plt.subplots(figsize=(8, 3.2))
bars = ax.barh(ty_le.index, ty_le.values, color=XANH, height=0.55)
ax.bar_label(bars, [f" {v:.0f}%" for v in ty_le.values])
ax.set_title("4 trên 5 listing Santiago là nguyên căn", loc="left", fontweight="bold")
ax.set_xlabel("% tổng số listing (n = 18.534)")
ax.grid(axis="y", alpha=0)
fig.savefig("figures/ty_le_loai_phong.png", dpi=150, bbox_inches="tight")
plt.show()

### Bài 2 — So sánh công bằng hai snapshot

Tải thêm bản 09/2025 (`{BASE_T9}/visualisations/listings.csv` với
`BASE_T9 = ".../santiago/2025-09-27"`). Vẽ 2 hist giá (log10) cạnh nhau cho 2 snapshot với
**cùng thang trục** (`sharex=True, sharey=True`). Phân phối giá có dịch chuyển không?

In [ ]:
# TODO Bài 2:
t9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/listings.csv")
g9 = t9.loc[t9["price"] > 0, "price"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True, sharey=True)
axes[0].hist(np.log10(g9), bins=50, color=XANH)
axes[0].set_title("09/2025")
axes[1].hist(np.log10(gia), bins=50, color=XANH)
axes[1].set_title("06/2026")
fig.suptitle("Phân phối log-giá gần như giữ nguyên hình dáng sau 9 tháng", fontweight="bold")
plt.tight_layout(); plt.show()

### Bài 3 — Chọn dạng cho 3 câu hỏi

Với mỗi câu hỏi, chọn dạng biểu đồ + vẽ (mỗi hình ≤ 8 dòng code):

1. "Số listing mỗi quận top 10 chênh nhau thế nào?"
2. "Điểm `reviews_per_month` phân bố ra sao (đa số im ắng hay đều đặn)?"
3. "Phòng nhiều review gần đây (`number_of_reviews_ltm`) có xu hướng rẻ hơn không?"
   (gợi ý: scatter + thang log cho giá; đừng quên alpha)

In [ ]:
# TODO Bài 3 — câu 3 làm mẫu:
s3 = df.dropna(subset=["price"])
s3 = s3[s3["price"] > 0].sample(5000, random_state=2)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.scatter(s3["number_of_reviews_ltm"], s3["price"], s=5, alpha=0.3, color=XANH)
ax.set_yscale("log")
ax.set_xlabel("số review 12 tháng qua"); ax.set_ylabel("giá (CLP, log)")
ax.set_title("Phòng bận rộn hiếm khi là phòng đắt nhất", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Thử thách về nhà 🏆 — Bộ 4 hình đầu tiên cho bài tập lớn

Với thành phố nhóm bạn: làm đúng **4 hình bắt buộc kiểu bài tập lớn** — (1) chuỗi thời gian review
đa snapshot, (2) bar xếp hạng quận theo một KPI, (3) phân phối giá (log), (4) scatter toạ độ.
Mỗi hình: qua checklist 4 điểm + lưu vào `figures/` + 2 câu diễn giải viết trong cell Markdown
ngay dưới hình. Nộp notebook + 4 file PNG.

In [ ]:
RUN_CHALLENGE = False
if RUN_CHALLENGE:
    CITY_BASE = "https://data.insideairbnb.com/..."   # thành phố nhóm bạn
    ...

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| Dạng hình đi theo câu hỏi; Anscombe: phải vẽ mới thấy | Chọn đúng vũ khí trước khi bắn |
| `fig, ax` + title thông điệp + đơn vị + nguồn | Hình tự giải thích được — chuẩn báo cáo bài tập lớn |
| Nhấn 1 thứ bằng màu; bar ngang cho tên dài; alpha cho scatter đông | Tín hiệu > trang trí |
| Bar từ 0; so sánh chung thang; mỗi hình một thông điệp | Trung thực thị giác |
| `rcParams` + `savefig` vào `figures/` từ pipeline | Phong cách đồng bộ, tái lập được |

**Buổi sau:** seaborn cho hình thống kê nhiều chiều, bản đồ có ranh giới quận — và mổ xẻ
những biểu đồ do AI sinh ra.